# 3 · Train / Validation / Test split (per timeframe, chronological, no shuffle)

Each TF's feature file is split independently using the **same calendar cutoffs**, so train/val/test
boundaries line up in time across D1/H4/H1/M15 even though row counts differ per TF.

| Split | Range |
|-------|-------|
| Train | `< 2022-01-01` |
| Val   | `2022-01-01` .. `2024-01-01` |
| Test  | `>= 2024-01-01` |

No shuffling — this is time series, so a random split would leak future information into training.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from config.settings import DATA_FEATURES, ROOT

DATA_SPLITS = ROOT / "data" / "splits"
DATA_SPLITS.mkdir(parents=True, exist_ok=True)

SYMBOL = "GBPUSD"
TIMEFRAMES = ["D1", "H4", "H1", "M15"]

TRAIN_END = pd.Timestamp("2022-01-01", tz="UTC")  # exclusive
VAL_END = pd.Timestamp("2024-01-01", tz="UTC")  # exclusive

features = {tf: pd.read_parquet(DATA_FEATURES / f"{SYMBOL}_{tf}_features.parquet") for tf in TIMEFRAMES}
for tf, df in features.items():
    print(f"{tf}: {len(df)} rows  ({df['datetime'].iloc[0]} .. {df['datetime'].iloc[-1]})")

D1: 3783 rows  (2012-01-11 00:00:00+00:00 .. 2026-07-10 00:00:00+00:00)
H4: 22680 rows  (2012-01-11 00:00:00+00:00 .. 2026-07-10 00:00:00+00:00)
H1: 90426 rows  (2012-01-11 01:00:00+00:00 .. 2026-07-10 00:00:00+00:00)
M15: 356499 rows  (2012-01-11 01:30:00+00:00 .. 2026-07-10 00:00:00+00:00)


## Split + save

In [2]:
def split_by_date(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    dt = df["datetime"]
    return {
        "train": df[dt < TRAIN_END].reset_index(drop=True),
        "val": df[(dt >= TRAIN_END) & (dt < VAL_END)].reset_index(drop=True),
        "test": df[dt >= VAL_END].reset_index(drop=True),
    }


for tf, df in features.items():
    splits = split_by_date(df)
    for name, part in splits.items():
        path = DATA_SPLITS / f"{SYMBOL}_{tf}_{name}.parquet"
        part.to_parquet(path, index=False)
        pct = len(part) / len(df) * 100
        span = f"{part['datetime'].iloc[0]} .. {part['datetime'].iloc[-1]}" if len(part) else "empty"
        print(f"[SAVE] {path.relative_to(ROOT)}: {len(part)} rows ({pct:.1f}%)  {span}")
    print()

[SAVE] data/splits/GBPUSD_D1_train.parquet: 2603 rows (68.8%)  2012-01-11 00:00:00+00:00 .. 2021-12-31 00:00:00+00:00
[SAVE] data/splits/GBPUSD_D1_val.parquet: 520 rows (13.7%)  2022-01-03 00:00:00+00:00 .. 2023-12-29 00:00:00+00:00
[SAVE] data/splits/GBPUSD_D1_test.parquet: 660 rows (17.4%)  2024-01-01 00:00:00+00:00 .. 2026-07-10 00:00:00+00:00

[SAVE] data/splits/GBPUSD_H4_train.parquet: 15608 rows (68.8%)  2012-01-11 00:00:00+00:00 .. 2021-12-31 20:00:00+00:00
[SAVE] data/splits/GBPUSD_H4_val.parquet: 3120 rows (13.8%)  2022-01-03 00:00:00+00:00 .. 2023-12-29 20:00:00+00:00
[SAVE] data/splits/GBPUSD_H4_test.parquet: 3952 rows (17.4%)  2024-01-01 00:00:00+00:00 .. 2026-07-10 00:00:00+00:00

[SAVE] data/splits/GBPUSD_H1_train.parquet: 62219 rows (68.8%)  2012-01-11 01:00:00+00:00 .. 2021-12-31 23:00:00+00:00
[SAVE] data/splits/GBPUSD_H1_val.parquet: 12470 rows (13.8%)  2022-01-03 00:00:00+00:00 .. 2023-12-29 23:00:00+00:00
[SAVE] data/splits/GBPUSD_H1_test.parquet: 15737 rows (17.4%)

## Sanity check — no overlap, no gaps, chronological order preserved

In [3]:
for tf in TIMEFRAMES:
    train = pd.read_parquet(DATA_SPLITS / f"{SYMBOL}_{tf}_train.parquet")
    val = pd.read_parquet(DATA_SPLITS / f"{SYMBOL}_{tf}_val.parquet")
    test = pd.read_parquet(DATA_SPLITS / f"{SYMBOL}_{tf}_test.parquet")

    assert train["datetime"].max() < val["datetime"].min(), f"{tf}: train/val overlap"
    assert val["datetime"].max() < test["datetime"].min(), f"{tf}: val/test overlap"
    assert train["datetime"].is_monotonic_increasing, f"{tf}: train not sorted"
    assert val["datetime"].is_monotonic_increasing, f"{tf}: val not sorted"
    assert test["datetime"].is_monotonic_increasing, f"{tf}: test not sorted"

    print(f"[OK] {tf}: train < val < test, chronological, no overlap")

[OK] D1: train < val < test, chronological, no overlap
[OK] H4: train < val < test, chronological, no overlap
[OK] H1: train < val < test, chronological, no overlap
[OK] M15: train < val < test, chronological, no overlap
